# XAI Engine — Integration Notebook

**Runtime logic only.** This notebook contains the same validated mechanism as the original
research notebook — simulate a change, re-query the real model, measure the shift — with every
research/evaluation step removed. No fidelity testing, no bootstrap confidence intervals, no
cross-model comparison. Those were one-time validation steps, not something an API should run
per request.

**Before you integrate this:**
1. Place this notebook in the same folder as your six `.joblib` files (three models + three
   metadata files) and `Preprocessed_Dataset.csv`.
2. Run every cell top to bottom. Section 5 gives you real, runnable test cases — confirm they
   produce sensible output before lifting any of this into your endpoint.
3. This was written and reasoned through carefully, but **has not been executed in this
   session** — the environment that had your actual model files reset before this notebook was
   built. Treat Section 5's results as the first real test, not a formality.

**Not clinical advice.** This reveals what would change a statistical prediction under a given
model — it carries no notion of test cost, feasibility, or clinical risk.

## 1. Setup

In [4]:
import logging
from pathlib import Path
from typing import Any, Optional

import joblib
import numpy as np
import pandas as pd
from scipy.stats import entropy

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("xai_engine")

DEFAULT_K_NEIGHBORS = 30
DEFAULT_MAX_CANDIDATES = 4
EPISTEMIC_THRESHOLD_BITS = 0.15

SUPPORTED_MODELS = ("discharge", "transfer", "stay")

MODEL_FILES = {
    "discharge": ("discharge_outcome_model.joblib", "discharge_outcome_metadata.joblib"),
    "transfer": ("transfer_model.joblib", "transfer_metadata.joblib"),
    "stay": ("stay_duration_model.joblib", "stay_duration_metadata.joblib"),
}

DATA_PATH = "Preprocessed_Dataset.csv"
MODEL_DIR = "artifacts"

print("Setup complete.")

Setup complete.


## 2. Errors

Typed exceptions so a bad request surfaces as a clean, specific error — not a raw stack trace —
once this is behind an API.

In [5]:
class XAIEngineError(Exception):
    """Raised for any XAI engine failure that should surface as a clean API error."""


class UnknownModelError(XAIEngineError):
    """Requested model_key is not one of the three supported models."""


class ModelLoadError(XAIEngineError):
    """A required model or metadata file could not be loaded."""


print("Error types defined.")

Error types defined.


## 3. Loading models and training data

Loads all three models and metadata once. Rebuilds `total_days` from the ward-day columns —
identical logic to the Stay Duration and Discharge Outcome notebooks, so the training pool used
for finding "similar patients" matches exactly what those models were trained on.

Every load step validates what it expects to find and raises a specific, readable error if a
file or column is missing, rather than failing deep inside a later calculation.

In [6]:
def load_models(model_dir=MODEL_DIR):
    models = {}
    model_dir = Path(model_dir)
    for key, (model_file, meta_file) in MODEL_FILES.items():
        model_path = model_dir / model_file
        meta_path = model_dir / meta_file
        if not model_path.exists() or not meta_path.exists():
            raise ModelLoadError(
                f"Missing model files for '{key}': expected {model_path} and {meta_path}")
        try:
            model = joblib.load(model_path)
            meta = joblib.load(meta_path)
        except Exception as exc:
            raise ModelLoadError(f"Failed to load model '{key}': {exc}") from exc

        required_meta_keys = {"features", "cat_features", "num_features",
                              "age_order", "time_order", "rare_maps"}
        missing = required_meta_keys - meta.keys()
        if missing:
            raise ModelLoadError(f"Metadata for '{key}' is missing keys: {missing}")

        models[key] = {"model": model, **meta}
        logger.info("Loaded model '%s' (%d features)", key, len(meta["features"]))
    return models


def consolidate_rare(frame, meta):
    out = frame.copy()
    for col, keep in meta.get("rare_maps", {}).items():
        if col in out.columns:
            keep_set = set(keep)
            out[col] = out[col].where(out[col].isna() | out[col].isin(keep_set), "other")
    return out


def load_training_pools(models, data_path=DATA_PATH):
    data_path = Path(data_path)
    if not data_path.exists():
        raise ModelLoadError(f"Training data not found at {data_path}")

    df_raw = pd.read_csv(data_path)
    for c in df_raw.columns:
        if df_raw[c].dtype == object:
            df_raw[c] = (df_raw[c].astype(str).str.strip().str.lower()
                        .replace({"nan": np.nan, "none": np.nan}))
    df_raw = df_raw.rename(columns={"Collision With ": "Collision With"})

    ward_cols = ["Number of days in first ward", "Number of days in Second ward",
                "Number of days in Third ward"]
    missing_ward_cols = [c for c in ward_cols if c not in df_raw.columns]
    if missing_ward_cols:
        raise ModelLoadError(f"Expected ward-day columns not found: {missing_ward_cols}")
    for c in ward_cols:
        df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")
        reason_col = f"{c}_missing_reason"
        if reason_col in df_raw.columns:
            early = df_raw[reason_col] == "early_discharge"
            df_raw.loc[early, c] = df_raw.loc[early, c].fillna(0)
    df_raw["total_days"] = df_raw[ward_cols].sum(axis=1, min_count=1)

    pools = {}
    for key, meta in models.items():
        missing_features = [f for f in meta["features"] if f not in df_raw.columns]
        if missing_features:
            raise ModelLoadError(f"Model '{key}' expects missing features: {missing_features}")
        pool = df_raw[meta["features"]].copy()
        pool = consolidate_rare(pool, meta)
        pools[key] = pool
        logger.info("Loaded training pool for '%s': %d rows", key, len(pool))
    return pools


MODELS = load_models()
TRAIN_POOLS = load_training_pools(MODELS)
print(f"\nReady. Models loaded: {list(MODELS.keys())}")

INFO:xai_engine:Loaded model 'discharge' (18 features)
INFO:xai_engine:Loaded model 'transfer' (21 features)
INFO:xai_engine:Loaded model 'stay' (8 features)
INFO:xai_engine:Loaded training pool for 'discharge': 6178 rows
INFO:xai_engine:Loaded training pool for 'transfer': 6178 rows
INFO:xai_engine:Loaded training pool for 'stay': 6178 rows



Ready. Models loaded: ['discharge', 'transfer', 'stay']


## 4. Core mechanism

Four pieces, same logic validated in the original research notebook:

1. **`predict_proba_row`** — calls a model's own pipeline exactly as it was trained, with ordinal
   encoding applied via that model's own stored `age_order`/`time_order`.
2. **`gower_distance`** — finds "similar patients": simple matching for categorical fields,
   normalised difference for the two ordinal fields. Missing values on either side are excluded
   from that feature's term, not penalised as maximally different.
3. **`simulate_missing_field`** — Value of Information. Samples plausible values for a missing
   field from similar patients, re-predicts for each, measures the entropy change.
4. **`simulate_known_field`** — counterfactual sensitivity. Samples plausible *alternative*
   values for a known field, re-predicts, measures the confidence shift and whether it flips the
   prediction.

In [7]:
def encode_ordinal(frame, meta):
    out = frame.copy()
    for col, order in (("Age", meta["age_order"]),
                      ("Time Taken To Reach Hospital", meta["time_order"])):
        if col in out.columns:
            out[col] = out[col].map({v: i for i, v in enumerate(order)})
    return out


def predict_proba_row(model_key, record):
    meta = MODELS[model_key]
    row = pd.DataFrame([{f: record.get(f) for f in meta["features"]}])
    row = encode_ordinal(row, meta)
    try:
        proba = meta["model"].predict_proba(row)[0]
    except Exception as exc:
        raise XAIEngineError(f"Model '{model_key}' failed to score the given record: {exc}") from exc
    classes = list(meta["model"].classes_)
    return proba, classes


def entropy_bits(proba):
    return float(entropy(proba, base=2))


def gower_distance(model_key, record, exclude_field=None):
    meta = MODELS[model_key]
    pool = TRAIN_POOLS[model_key]
    features = [f for f in meta["features"] if f != exclude_field]
    ord_cols = {"Age": meta["age_order"], "Time Taken To Reach Hospital": meta["time_order"]}

    n = len(pool)
    total_dist = np.zeros(n)
    total_weight = np.zeros(n)

    for f in features:
        rv = record.get(f)
        if rv is None or (isinstance(rv, float) and pd.isna(rv)):
            continue
        if f in ord_cols:
            order = ord_cols[f]
            order_map = {v: i for i, v in enumerate(order)}
            if rv not in order_map:
                continue
            rv_idx = order_map[rv]
            pool_idx = pool[f].map(order_map)
            known = pool_idx.notna().values
            d = (pool_idx - rv_idx).abs().values / (len(order) - 1)
            total_dist[known] += np.nan_to_num(d[known])
            total_weight[known] += 1.0
        else:
            pool_vals = pool[f].values
            known = pd.notna(pool_vals)
            d = np.where(pool_vals == rv, 0.0, 1.0)
            total_dist[known] += d[known]
            total_weight[known] += 1.0

    with np.errstate(invalid="ignore", divide="ignore"):
        avg = np.where(total_weight > 0, total_dist / np.maximum(total_weight, 1), np.inf)
    return avg, total_weight


def plausible_values(model_key, record, field, k=DEFAULT_K_NEIGHBORS,
                     max_candidates=DEFAULT_MAX_CANDIDATES, exclude_current=False):
    d, w = gower_distance(model_key, record, exclude_field=field)
    valid = w >= 1
    pool = TRAIN_POOLS[model_key]
    idx_sorted = np.argsort(np.where(valid, d, np.inf))
    neighbor_idx = idx_sorted[:k]
    neighbor_vals = pool[field].iloc[neighbor_idx].dropna()

    if exclude_current:
        current = record.get(field)
        neighbor_vals = neighbor_vals[neighbor_vals != current]

    if len(neighbor_vals) == 0:
        return []

    counts = neighbor_vals.value_counts(normalize=True)
    top = counts.head(max_candidates)
    return list(zip(top.index.tolist(), top.values.tolist()))


# Self-check: distance from a patient to themself must be exactly 0
_sample = TRAIN_POOLS["discharge"].iloc[0].to_dict()
_d, _w = gower_distance("discharge", _sample)
assert _d[0] == 0.0, "self-distance should be 0"
print("Core prediction and distance functions loaded; self-distance check passed.")

Core prediction and distance functions loaded; self-distance check passed.


In [8]:
def simulate_missing_field(model_key, record, field, k=DEFAULT_K_NEIGHBORS,
                           max_candidates=DEFAULT_MAX_CANDIDATES):
    current_proba, _ = predict_proba_row(model_key, record)
    current_entropy = entropy_bits(current_proba)

    candidates = plausible_values(model_key, record, field, k=k, max_candidates=max_candidates)
    if not candidates:
        return None

    sim_entropies, weights = [], []
    for value, weight in candidates:
        modified = dict(record)
        modified[field] = value
        proba, _ = predict_proba_row(model_key, modified)
        sim_entropies.append(entropy_bits(proba))
        weights.append(weight)

    weighted_avg_entropy = float(np.average(sim_entropies, weights=weights))
    return {"field": field, "current_entropy_bits": round(current_entropy, 4),
            "expected_post_entropy_bits": round(weighted_avg_entropy, 4),
            "expected_reduction_bits": round(current_entropy - weighted_avg_entropy, 4),
            "n_candidates_simulated": len(candidates)}


def simulate_known_field(model_key, record, field, k=DEFAULT_K_NEIGHBORS,
                         max_candidates=DEFAULT_MAX_CANDIDATES):
    current_proba, classes = predict_proba_row(model_key, record)
    pred_idx = int(np.argmax(current_proba))
    pred_class, pred_prob = classes[pred_idx], float(current_proba[pred_idx])

    candidates = plausible_values(model_key, record, field, k=k, max_candidates=max_candidates,
                                  exclude_current=True)
    if not candidates:
        return None

    results = []
    for value, weight in candidates:
        modified = dict(record)
        modified[field] = value
        proba, _ = predict_proba_row(model_key, modified)
        new_idx = int(np.argmax(proba))
        confidence_drop = pred_prob - float(proba[pred_idx])
        results.append({"alternative_value": value, "resulting_prediction": classes[new_idx],
                        "confidence_drop": round(confidence_drop, 4),
                        "confidence_shift_abs": round(abs(confidence_drop), 4),
                        "flips_prediction": classes[new_idx] != pred_class})

    # Rank by magnitude of shift, not raw signed value - a field whose alternatives all
    # INCREASE confidence (negative confidence_drop) is just as sensitive as one whose
    # alternatives all decrease it. Using plain max() over signed values would understate
    # sensitivity whenever every alternative points the same (increasing) direction.
    most_sensitive = max(results, key=lambda r: r["confidence_shift_abs"])

    return {"field": field, "current_prediction": pred_class,
            "current_probability": round(pred_prob, 4), "alternatives": results,
            "max_confidence_shift": most_sensitive["confidence_shift_abs"],
            "most_sensitive_alternative": most_sensitive["alternative_value"],
            "any_flip": any(r["flips_prediction"] for r in results)}


print("Simulation functions loaded.")

Simulation functions loaded.


## 5. The `explain` function — this is the endpoint's payload

This is the single function your API route should call. It returns one JSON-serialisable dict —
this is the contract to hand to whoever builds the FastAPI route.

In [9]:
def explain(model_key, record, k=DEFAULT_K_NEIGHBORS, max_candidates=DEFAULT_MAX_CANDIDATES,
           epistemic_threshold_bits=EPISTEMIC_THRESHOLD_BITS):
    if model_key not in SUPPORTED_MODELS:
        raise UnknownModelError(f"'{model_key}' is not supported. Valid: {SUPPORTED_MODELS}")

    meta = MODELS[model_key]
    features = meta["features"]
    missing_fields = [f for f in features
                      if record.get(f) is None or (isinstance(record.get(f), float)
                                                   and pd.isna(record.get(f)))]
    known_fields = [f for f in features if f not in missing_fields]

    proba, classes = predict_proba_row(model_key, record)
    pred_idx = int(np.argmax(proba))
    pred_class, pred_prob = classes[pred_idx], float(proba[pred_idx])
    current_entropy = entropy_bits(proba)
    max_entropy = float(np.log2(len(classes)))

    result = {"model": model_key, "predicted_class": pred_class,
              "predicted_probability": round(pred_prob, 4),
              "entropy_bits": round(current_entropy, 4), "max_entropy_bits": round(max_entropy, 4),
              "n_missing_fields": len(missing_fields), "n_known_fields": len(known_fields)}

    if missing_fields:
        voi_results = []
        for f in missing_fields:
            r = simulate_missing_field(model_key, record, f, k=k, max_candidates=max_candidates)
            if r is not None:
                voi_results.append(r)
        voi_results.sort(key=lambda r: r["expected_reduction_bits"], reverse=True)
        result["value_of_information_ranking"] = voi_results

        if voi_results and voi_results[0]["expected_reduction_bits"] >= epistemic_threshold_bits:
            result["uncertainty_type"] = "epistemic"
            result["recommended_field"] = voi_results[0]["field"]
            result["message"] = (f"Uncertain ({pred_class}, {pred_prob:.0%}). Getting "
                                 f"'{voi_results[0]['field']}' would likely help most "
                                 f"(expected reduction: {voi_results[0]['expected_reduction_bits']:.2f} bits).")
        else:
            result["uncertainty_type"] = "aleatoric"
            result["message"] = (f"Uncertain ({pred_class}, {pred_prob:.0%}). No missing field is "
                                 f"expected to meaningfully resolve this - the case appears "
                                 f"genuinely borderline.")
    else:
        result["uncertainty_type"] = "not_applicable_fully_specified"
        result["value_of_information_ranking"] = []

    if known_fields:
        cf_results = []
        for f in known_fields:
            r = simulate_known_field(model_key, record, f, k=k, max_candidates=max_candidates)
            if r is not None:
                cf_results.append(r)
        cf_results.sort(key=lambda r: r["max_confidence_shift"], reverse=True)
        result["counterfactual_ranking"] = cf_results
        if cf_results:
            top = cf_results[0]
            result["hinge_field"] = top["field"]
            flip_note = ""
            if top["any_flip"]:
                flipped_to = next(a["resulting_prediction"] for a in top["alternatives"]
                                  if a["flips_prediction"])
                flip_note = f" A plausible alternative value would flip it to '{flipped_to}'."
            result["hinge_message"] = (f"This prediction is most sensitive to '{top['field']}' - "
                                       f"an alternative plausible value would shift confidence by "
                                       f"up to {top['max_confidence_shift']:.0%}.{flip_note}")
    else:
        result["counterfactual_ranking"] = []

    return result


print("explain() defined - this is the function your API route should call.")

explain() defined - this is the function your API route should call.


## 6. Test before you integrate

**Run this section for real** — this is the first actual execution of this logic since it was
rewritten for integration. Confirm the output looks sensible before wiring `explain()` into a
FastAPI route.

Three cases: a sparse record (should trigger epistemic or aleatoric), a fully-specified record
(should skip straight to counterfactual sensitivity), and an intentionally bad `model_key`
(should raise `UnknownModelError` cleanly, not crash).

In [10]:
import json as _json

print("=== Test 1: sparse record ===")
complete = TRAIN_POOLS["discharge"].dropna()
full_record = complete.iloc[0].to_dict()
sparse_record = {"Age": full_record["Age"], "Site of Injury No1": full_record["Site of Injury No1"]}
result_1 = explain("discharge", sparse_record)
print(f"predicted: {result_1['predicted_class']} ({result_1['predicted_probability']:.0%})")
print(f"uncertainty_type: {result_1['uncertainty_type']}")
print(result_1.get("message", ""))

print("\n=== Test 2: fully-specified record ===")
result_2 = explain("transfer", TRAIN_POOLS["transfer"].dropna().iloc[0].to_dict())
print(f"predicted: {result_2['predicted_class']} ({result_2['predicted_probability']:.0%})")
print(f"uncertainty_type: {result_2['uncertainty_type']}")
print(result_2.get("hinge_message", "(no hinge field found)"))

print("\n=== Test 3: bad model_key raises cleanly ===")
try:
    explain("not_a_real_model", {})
    print("FAIL: should have raised UnknownModelError")
except UnknownModelError as exc:
    print(f"Raised correctly: {exc}")

print("\n=== Full JSON payload for Test 1 (this is what your endpoint would return) ===")
print(_json.dumps(result_1, indent=2, default=str))

=== Test 1: sparse record ===
predicted: complete recovery (48%)
uncertainty_type: epistemic
Uncertain (complete recovery, 48%). Getting 'Type of injury No 1' would likely help most (expected reduction: 0.22 bits).

=== Test 2: fully-specified record ===
predicted: yes (74%)
uncertainty_type: not_applicable_fully_specified
This prediction is most sensitive to 'Time Taken To Reach Hospital' - an alternative plausible value would shift confidence by up to 14%.

=== Test 3: bad model_key raises cleanly ===
Raised correctly: 'not_a_real_model' is not supported. Valid: ('discharge', 'transfer', 'stay')

=== Full JSON payload for Test 1 (this is what your endpoint would return) ===
{
  "model": "discharge",
  "predicted_class": "complete recovery",
  "predicted_probability": 0.4825,
  "entropy_bits": 1.5164,
  "max_entropy_bits": 1.585,
  "n_missing_fields": 16,
  "n_known_fields": 2,
  "value_of_information_ranking": [
    {
      "field": "Type of injury No 1",
      "current_entropy_bits"

## 7. Handoff notes for the API layer

- Call `explain(model_key, record)` once per request. Do not re-run Sections 1–4 per request —
  load models and training pools once, at application startup.
- `model_key` must be one of `"discharge"`, `"transfer"`, `"stay"` — anything else raises
  `UnknownModelError`, catch it and return an HTTP 400.
- A malformed record that the model itself can't score raises `XAIEngineError` — catch it and
  return an HTTP 422, not a 500.
- The FastAPI router built alongside this notebook (`xai_router.py`) already wraps this in a
  `POST /explain` endpoint with request/response schemas, if you want to use it directly instead
  of rewriting the wiring.
- This is a research prototype. It is not clinical advice, and it carries no notion of test
  cost, feasibility, or clinical risk — that framing should survive into whatever UI eventually
  calls this endpoint.